# EndoScan — Scan Router (Modality Detection + Dispatch)

**What this notebook is:** the single entry point a user's uploaded scan goes
through. It does NOT retrain anything — it loads the two already-trained
checkpoints (MRI, laparoscopy) produced by their respective training
notebooks, detects which modality an uploaded image is, and routes it to the
correct model, Grad-CAM, and GenAI explanation — all as a **complementary
explainer**, matching the corrected role from the MRI notebook: this never
diagnoses or scores severity, and nothing here feeds the fusion layer.

**Modality detection approach:** rather than training a third classifier
just to tell MRI from laparoscopy apart (expensive, and unnecessary given
how different these two image types actually are), this uses a simple,
robust heuristic — laparoscopy frames are real colour video (genuine
variation across R/G/B channels), while the MRI pipeline saves grayscale
slices as three *identical* stacked channels. Checking channel variance is
cheap, deterministic, and fully explainable if it ever misroutes — unlike a
learned classifier trained on limited modality-labelled data.

**Known asymmetry to flag to your mentor:** the laparoscopy model, as
currently trained, doesn't have the calibration or threshold-tuning that the
MRI path has (Phase 5 in the MRI notebook) — it still uses a flat 0.5
decision threshold and raw uncalibrated probabilities. This notebook uses it
as-is rather than silently pretending both paths are equally rigorous; the
laparoscopy notebook should get the same calibration + threshold-tuning
treatment before this is genuinely production-ready. This notebook is
extensible to a third modality (ultrasound) once that data exists — see the
`MODALITY_CONFIG` structure in Cell 3.

In [1]:
# ── Cell 1 — Imports ─────────────────────────────────────────────
!pip install opencv-python-headless google-generativeai -q

import os
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import timm
import cv2
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cpu


**What this cell does:** Standard imports. Nothing modality-specific
yet — this is just the shared toolkit both paths below need (PyTorch, timm
for the EfficientNet-B0 architecture, OpenCV for the Grad-CAM overlay).

In [2]:
# ── Cell 2 — Model architecture (shared by both modalities) ─────────
# Both the MRI and laparoscopy models use the same EfficientNet-B0 backbone
# with only the last block + head unfrozen, and the same classifier head —
# this function builds the untrained architecture; the checkpoint's
# state_dict is loaded into it afterward. Building it fresh (rather than
# trying to share one in-memory model object) keeps the two paths fully
# independent, which matters since they were trained separately and their
# weights are not compatible with each other.
def build_efficientnet_b0_head(dropout: float = 0.3) -> nn.Module:
    model = timm.create_model('efficientnet_b0', pretrained=False)  # weights come from checkpoint
    in_feat = model.classifier.in_features
    model.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_feat, 1))
    return model.to(DEVICE)

IMG_SIZE = 224
print("Architecture builder defined ✓")

Architecture builder defined ✓


**What this cell does:** Defines the shared model architecture builder.
Both trained models are EfficientNet-B0 with the same classifier head
(dropout + single linear output), so one function builds either — the
actual learned weights come from each modality's own checkpoint file in the
next cell, not from ImageNet pretraining (`pretrained=False` here, since
we're about to overwrite every weight anyway).

In [4]:
# ── Cell 3 — Load both checkpoints + modality config ────────────────
# ── UPDATE THESE PATHS to match where your two trained checkpoints live ──
MRI_CHECKPOINT_PATH   = Path('./outputs/mri_best_model.pt')
MRI_CALIBRATION_PATH  = Path('./outputs/mri_calibration.pkl')
LAPARO_CHECKPOINT_PATH = Path('./outputs/efficientnet_b0_endoscan_final.pt')
# ──────────────────────────────────────────────────────────────────

import joblib

assert MRI_CHECKPOINT_PATH.exists(), f"MRI checkpoint not found at {MRI_CHECKPOINT_PATH}"
assert LAPARO_CHECKPOINT_PATH.exists(), f"Laparoscopy checkpoint not found at {LAPARO_CHECKPOINT_PATH}"

# ── MRI model ─────────────────────────────────────────────────────
mri_model = build_efficientnet_b0_head()
mri_ckpt = torch.load(MRI_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
mri_model.load_state_dict(mri_ckpt['model_state_dict'])
mri_model.eval()
mri_threshold = mri_ckpt.get('chosen_threshold', 0.5)

mri_platt = None
if MRI_CALIBRATION_PATH.exists():
    cal = joblib.load(MRI_CALIBRATION_PATH)
    mri_platt = cal['platt']
    mri_threshold = cal.get('chosen_threshold', mri_threshold)
print(f"MRI model loaded ✓  threshold={mri_threshold:.2f}  calibrated={mri_platt is not None}")

# ── Laparoscopy model ────────────────────────────────────────────
laparo_model = build_efficientnet_b0_head()
laparo_ckpt = torch.load(LAPARO_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
laparo_model.load_state_dict(laparo_ckpt['model_state_dict'])
laparo_model.eval()
# ⚠ No calibration or tuned threshold exists for this model yet — flat 0.5.
# Flagged in the intro markdown above; fix by adding a Phase 5 (calibration
# + threshold tuning) to the laparoscopy notebook, matching the MRI one.
laparo_threshold = 0.5
laparo_platt = None
print(f"Laparoscopy model loaded ✓  threshold={laparo_threshold:.2f}  "
      f"calibrated={laparo_platt is not None}  ⚠ uncalibrated, flat threshold")

# ── Per-modality config — add 'ultrasound' here once that data/model exists ──
MODALITY_CONFIG = {
    'mri': {
        'model': mri_model,
        'threshold': mri_threshold,
        'platt': mri_platt,
        'grad_cam_target_layer': lambda m: m.blocks[-1][-1],
        'finding_labels': {
            1: 'Possible endometriosis indicators visible on this MRI slice',
            0: 'No endometriosis indicators visible on this MRI slice',
        },
        'scan_description': 'MRI slice',
    },
    'laparoscopy': {
        'model': laparo_model,
        'threshold': laparo_threshold,
        'platt': laparo_platt,
        'grad_cam_target_layer': lambda m: m.blocks[6],
        'finding_labels': {
            1: 'Possible endometriosis tissue visible on this laparoscopy frame',
            0: 'No endometriosis tissue visible on this laparoscopy frame',
        },
        'scan_description': 'laparoscopy image',
    },
    # 'ultrasound': { ... }  # not yet available — data collection pending
}

print(f"\nConfigured modalities: {list(MODALITY_CONFIG.keys())}")

MRI model loaded ✓  threshold=0.36  calibrated=True
Laparoscopy model loaded ✓  threshold=0.50  calibrated=False  ⚠ uncalibrated, flat threshold

Configured modalities: ['mri', 'laparoscopy']


**What this cell does:** Loads both trained checkpoints into separate,
independent model instances (never sharing weights between them — they were
trained on completely different tasks), plus the MRI path's Platt
calibrator and tuned threshold if that file exists. `MODALITY_CONFIG` is the
single place that defines everything modality-specific: which model to use,
its decision threshold, its Grad-CAM target layer, and the wording for its
finding labels — adding ultrasound later means adding one more entry here,
nothing else in this notebook needs to change. **Update the two checkpoint
paths at the top** to wherever your actual trained files live.

In [5]:
# ── Cell 4 — Modality detection ──────────────────────────────────
def detect_modality(image_path, channel_diff_threshold: float = 5.0) -> str:
    """
    Returns 'mri' or 'laparoscopy' based on colour-channel variance.
    MRI slices are saved as grayscale (R==G==B for every pixel, since the
    extraction pipeline stacks one channel three times). Laparoscopy frames
    are real colour video with genuine variation across channels.

    channel_diff_threshold: mean absolute difference between channels, in
    0-255 units, above which an image is considered colour. A small
    tolerance (not exactly 0) absorbs negligible PNG/JPEG compression noise
    in genuinely-grayscale images.
    """
    img = np.array(Image.open(image_path).convert('RGB'), dtype=np.float32)
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    mean_channel_diff = (np.abs(r - g).mean() + np.abs(g - b).mean() + np.abs(r - b).mean()) / 3
    return 'laparoscopy' if mean_channel_diff > channel_diff_threshold else 'mri'


print("detect_modality() defined ✓")
print(f"Channel-difference threshold: 5.0 (out of 255) — tune in Cell 4b if misrouting occurs")

detect_modality() defined ✓
Channel-difference threshold: 5.0 (out of 255) — tune in Cell 4b if misrouting occurs


**What this cell does:** The actual modality heuristic — computes the
average absolute difference between R/G/B channels across the whole image.
A true grayscale MRI PNG will have this at or near 0; a colour laparoscopy
frame will have real separation between channels (skin/tissue tones aren't
neutral grey). This is intentionally simple and inspectable — if it ever
misroutes an image, you can look at the actual channel-diff number and
understand exactly why, unlike a learned classifier's failure mode.

In [6]:
# ── Cell 4b — Validate the heuristic on real sample images ──────────
# Don't trust the threshold blind — check it against real images from both
# datasets before relying on it. Update these sample paths to a few real
# files from each of your PNG output folders.
SAMPLE_MRI_IMAGES = list(Path('./outputs/mri_slices_v2').glob('*.png'))[:5]
SAMPLE_LAPARO_IMAGES = list(Path('./Glenda_v1.5_classes/frames').glob('*.jpg'))[:5] \
                       if Path('./Glenda_v1.5_classes/frames').exists() else []

print("MRI samples (should all say 'mri'):")
for p in SAMPLE_MRI_IMAGES:
    print(f"  {p.name:<40} -> {detect_modality(p)}")

print("\nLaparoscopy samples (should all say 'laparoscopy'):")
for p in SAMPLE_LAPARO_IMAGES:
    print(f"  {p.name:<40} -> {detect_modality(p)}")

if not SAMPLE_LAPARO_IMAGES:
    print("\n⚠ No laparoscopy sample images found at the path above — update it "
          "to your actual GLENDA frames directory before trusting this check.")

MRI samples (should all say 'mri'):
  D2-050_sl057_label0.png                  -> mri
  D1-010_sl014_label0.png                  -> mri
  D1-020_sl019_label0.png                  -> mri
  D2-007_sl016_label0.png                  -> mri
  D1-047_sl014_label0.png                  -> mri

Laparoscopy samples (should all say 'laparoscopy'):
  c_42_v_(video_1323.mp4)_f_183.jpg        -> laparoscopy
  c_28_v_(video_907.mp4)_f_73.jpg          -> laparoscopy
  c_99_v_(video_3006.mp4)_f_102.jpg        -> laparoscopy
  c_3_v_(video_29.mp4)_f_480.jpg           -> laparoscopy
  c_65_v_(video_2087.mp4)_f_1648.jpg       -> laparoscopy


**What this cell does:** Sanity-checks the heuristic against a handful
of real images from both pipelines before trusting it in production.
**Update the two sample paths** to match your actual folders, run this, and
confirm every MRI sample says `mri` and every laparoscopy sample says
`laparoscopy`. If anything misroutes, print the actual channel-diff value
for that file (add a quick debug line) and adjust `channel_diff_threshold`
in Cell 4 accordingly — don't skip this step, since a wrong route sends an
image to a model that was never trained to interpret it.

In [7]:
# ── Cell 5 — Shared Grad-CAM (works for either model) ────────────────
def make_grad_cam(model: nn.Module, target_layer: nn.Module):
    """Returns a grad_cam(img_tensor) function hooked to the given layer."""
    activations, gradients = {}, {}
    def fwd_hook(module, inp, out): activations['feat'] = out.detach()
    def bwd_hook(module, g_in, g_out): gradients['feat'] = g_out[0].detach()
    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    def grad_cam(img_tensor):
        model.eval()
        img = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
        out = model(img)
        model.zero_grad(); out.backward()
        grads, acts = gradients['feat'].squeeze(0), activations['feat'].squeeze(0)
        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0).cpu().numpy()
        cam = np.maximum(cam, 0)
        if cam.max() > 0: cam /= cam.max()
        return np.array(Image.fromarray((cam * 255).astype(np.uint8))
                         .resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0

    return grad_cam, (h1, h2)


print("make_grad_cam() defined ✓ — works for either model, given its target layer")

make_grad_cam() defined ✓ — works for either model, given its target layer


**What this cell does:** A generalised version of the Grad-CAM logic
from the MRI notebook, parameterised by model and target layer instead of
being hardcoded to one model — so both paths reuse the exact same
implementation rather than maintaining two copies. `MODALITY_CONFIG`'s
`grad_cam_target_layer` lambda supplies the right layer for whichever
model gets used.

In [8]:
# ── Cell 6 — Unified routing + inference ─────────────────────────────
def preprocess_for_model(image_path, modality: str) -> torch.Tensor:
    """
    Each modality's training pipeline normalised images differently — this
    MUST match, or the model sees out-of-distribution input and its
    confidence numbers become meaningless. MRI: single-channel, manually
    scaled to [-1, 1], then stacked to 3 channels. Laparoscopy: real RGB,
    ImageNet mean/std normalisation (matching its albumentations pipeline).
    """
    if modality == 'mri':
        raw = np.array(Image.open(image_path).convert('L'), dtype=np.float32)
        sl = raw / 255.0 * 2.0 - 1.0
        if sl.shape[0] != IMG_SIZE or sl.shape[1] != IMG_SIZE:
            from scipy.ndimage import zoom as scipy_zoom
            sl = scipy_zoom(sl, (IMG_SIZE / sl.shape[0], IMG_SIZE / sl.shape[1]), order=1)
        tens = torch.tensor(np.stack([sl, sl, sl], 0), dtype=torch.float32)
        return tens
    else:  # laparoscopy
        img = np.array(Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_norm = (img - mean) / std
        tens = torch.tensor(img_norm.transpose(2, 0, 1), dtype=torch.float32)
        return tens


def route_and_explain(image_path) -> dict:
    """
    THE single entry point: takes an uploaded scan, detects modality, runs
    the correct model, and returns a finding + confidence + Grad-CAM
    payload. This is a complementary explainer — it never diagnoses or
    scores severity, and its output is never sent to the fusion layer.
    Only offered to patients who are already diagnosed and want their
    existing scan explained.
    """
    modality = detect_modality(image_path)
    cfg = MODALITY_CONFIG[modality]
    model = cfg['model']

    tens = preprocess_for_model(image_path, modality)
    with torch.no_grad():
        raw_prob = torch.sigmoid(model(tens.unsqueeze(0).to(DEVICE))).item()

    if cfg['platt'] is not None:
        confidence = float(cfg['platt'].predict_proba([[raw_prob]])[:, 1][0])
    else:
        confidence = raw_prob   # uncalibrated — see the ⚠ note in Cell 3

    finding = int(confidence > cfg['threshold'])

    grad_cam_fn, hooks = make_grad_cam(model, cfg['grad_cam_target_layer'](model))
    cam = grad_cam_fn(tens)
    hooks[0].remove(); hooks[1].remove()

    return {
        'modality':          modality,
        'finding':           cfg['finding_labels'][finding],
        'finding_positive':  bool(finding),
        'confidence':        round(confidence, 4),
        'confidence_calibrated': cfg['platt'] is not None,
        'threshold_used':    cfg['threshold'],
        'scan_description':  cfg['scan_description'],
        'cam':               cam,          # for the GenAI explanation cell
        'image_path':        str(image_path),
        'role':              'complementary_explainer',
        'used_by_fusion_layer': False,
        'disclaimer':        'This is an AI-generated explanation of an existing scan, '
                              'not a diagnosis. Always discuss results with your clinician.',
    }


print("route_and_explain() defined ✓ — the single entry point for an uploaded scan")

route_and_explain() defined ✓ — the single entry point for an uploaded scan


**What this cell does:** This is the actual router. `preprocess_for_model`
matters more than it looks — the MRI and laparoscopy training pipelines
normalised images completely differently (manual [-1,1] scaling for MRI vs
ImageNet mean/std for laparoscopy), and using the wrong preprocessing for a
given model silently produces garbage confidence numbers rather than an
obvious error, so getting this branch right per modality is essential.
`route_and_explain()` ties everything together: detect modality → pick the
right model, threshold, and calibrator from `MODALITY_CONFIG` → run
inference → generate Grad-CAM → return one consistent payload shape
regardless of which modality was actually used, so the GenAI explanation
cell next doesn't need to know or care which model produced it.

In [17]:
# ── Cell 7 — Shared GenAI explanation layer (Gemini) ─────────────────
import google.generativeai as genai

genai.configure(api_key='AQ.Ab8RN6JP5NjgcwNljMjfwQA5kMbTUq7O38zKUu4lOAOfsa8_vw')

GEMINI_MODEL_NAME = 'gemini-3.6-flash'  
gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)


def describe_attention_region(cam: np.ndarray, threshold: float = 0.6) -> str:
    h, w = cam.shape
    ys, xs = np.where(cam > threshold)
    if len(ys) == 0:
        return "no single strongly localised region"
    cy, cx = ys.mean() / h, xs.mean() / w
    vert = 'upper' if cy < 0.4 else ('lower' if cy > 0.6 else 'central')
    horiz = 'left' if cx < 0.4 else ('right' if cx > 0.6 else 'central')
    return 'central' if (vert == 'central' and horiz == 'central') else f'{vert}-{horiz}'


def explain_scan_for_patient(result: dict) -> str:
    """
    Turns a route_and_explain() payload into a plain-language explanation.
    One prompt template for BOTH modalities — the wording adapts via
    result['scan_description'] rather than needing a separate prompt per
    modality, matching how MODALITY_CONFIG avoids duplicating logic.
    """
    raw = np.array(Image.open(result['image_path']).convert('RGB'), dtype=np.float32)
    cam_resized = cv2.resize(result['cam'], (raw.shape[1], raw.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.55 * raw + 0.45 * heatmap).astype(np.uint8)
    overlay_img = Image.fromarray(overlay)

    region_desc = describe_attention_region(result['cam'])

    if result['confidence_calibrated']:
        confidence_line = f"Confidence : {result['confidence']*100:.0f}%"
    else:
        # Uncalibrated raw sigmoid output can hit extreme values (e.g. 100%)
        # that don't reflect genuine certainty — showing that number to a
        # patient (or in a demo) implies false precision. Use a qualitative
        # bucket instead until this model gets proper calibration.
        p = result['confidence']
        bucket = 'low' if p < 0.4 else ('moderate' if p < 0.75 else 'high')
        confidence_line = f"Confidence : {bucket} (this model's confidence score is not yet calibrated, so an exact percentage isn't shown)"

    prompt = f"""You are EndoScan AI, a compassionate assistant that helps patients
understand their own {result['scan_description']}. This patient has ALREADY
been diagnosed — your job is only to explain what THIS SPECIFIC SCAN shows,
in plain language. You are not making a new diagnosis and you are not
assigning a severity level.

The image shown is a {result['scan_description']} with a Grad-CAM heatmap
overlay. Red/yellow areas show where the AI model focused its attention.
Blue areas show regions the model largely ignored.

Model findings for this scan:
- Finding    : {result['finding']}
- {confidence_line}
- The model focused most on the {region_desc} area of the image

Write a warm, plain-English explanation of this scan for a patient with no
medical background. Rules:
- 3-4 short paragraphs, calm and reassuring tone
- Explain what the highlighted area means in simple terms
- Never use jargon without immediately explaining it
- Do not state or imply a new diagnosis or severity level — this only
  explains what this scan shows
- If this finding doesn't seem to match what their clinician told them,
  gently suggest they bring this explanation to their next appointment
- Always remind them this is an AI-generated explanation of an existing
  scan, not a diagnosis
- Encourage them to discuss it with their gynaecologist

Write only the explanation, nothing else."""

    response = gemini_model.generate_content([overlay_img, prompt])
    return response.text


print("explain_scan_for_patient() defined ✓ — one prompt template for both modalities")

explain_scan_for_patient() defined ✓ — one prompt template for both modalities


**What this cell does:** One shared GenAI explanation function for
both modalities, rather than two near-duplicate ones — the prompt adapts
via `result['scan_description']` ("MRI slice" vs "laparoscopy image") so
the wording stays natural without duplicating the prompt logic. It also
surfaces the calibration asymmetry directly in the prompt: if the
laparoscopy path's uncalibrated confidence is being explained, the model is
told to add a note that the percentage is approximate — better to be
explicit about that limitation than let the explanation imply equal
precision it doesn't have.

In [12]:
import os
print(os.listdir('outputs/mri_slices_v2')[:5])

['D2-050_sl057_label0.png', 'D1-010_sl014_label0.png', 'D1-020_sl019_label0.png', 'D2-007_sl016_label0.png', 'D1-047_sl014_label0.png']


In [13]:
print(os.listdir('Glenda_v1.5_classes/frames')[:5])

['c_42_v_(video_1323.mp4)_f_183.jpg', 'c_28_v_(video_907.mp4)_f_73.jpg', 'c_99_v_(video_3006.mp4)_f_102.jpg', 'c_3_v_(video_29.mp4)_f_480.jpg', 'c_65_v_(video_2087.mp4)_f_1648.jpg']


In [18]:
# ── Cell 8 — End-to-end test ──────────────────────────────────────
# Point this at one real MRI file and one real laparoscopy file to confirm
# the full pipeline — detection, routing, inference, explanation — works
# end to end for both modalities before wiring this into the actual app.
TEST_MRI_IMAGE = Path('./outputs/mri_slices_v2/D2-050_sl057_label0.png')
TEST_LAPARO_IMAGE = Path('./Glenda_v1.5_classes/frames/c_42_v_(video_1323.mp4)_f_183.jpg')

for label, test_path in [('MRI', TEST_MRI_IMAGE), ('Laparoscopy', TEST_LAPARO_IMAGE)]:
    if test_path is None:
        print(f"{label}: set TEST_{label.upper()}_IMAGE above to run this test")
        continue
    print(f"\n{'='*60}\n  {label} TEST\n{'='*60}")
    result = route_and_explain(test_path)
    print(f"Detected modality : {result['modality']}")
    print(f"Finding           : {result['finding']}")
    print(f"Confidence        : {result['confidence']*100:.0f}%  "
          f"(calibrated={result['confidence_calibrated']})")
    explanation = explain_scan_for_patient(result)
    print(f"\n{explanation}")


  MRI TEST
Detected modality : mri
Finding           : No endometriosis indicators visible on this MRI slice
Confidence        : 2%  (calibrated=True)

Hello! This image is a single cross-section slice from your MRI scan. To help you see how the computer model evaluated this image, a colorful overlay called a heatmap has been added. The bright red and yellow areas highlight where the AI focused most of its attention, while the blue areas show regions it largely passed over.

On this specific slice, the AI concentrated heavily on the central region, which includes tissue and blood vessels near the middle of your body. After examining this highlighted area, the AI model found no visual indicators of endometriosis on this particular slice image.

It is important to keep in mind that a full MRI is made up of many individual slices. Endometriosis tissue might be present on other slices that were taken, or it may be too subtle to appear on this single frame. If this specific result feels di

**What this cell does:** The full end-to-end smoke test — set both
`TEST_MRI_IMAGE` and `TEST_LAPARO_IMAGE` to a real file path from each
dataset and run this to confirm the entire pipeline works for both
modalities before this gets wired into the actual application. If either
one produces a wrong-looking finding or an obviously broken explanation,
that's a sign something upstream (usually the preprocessing branch or the
modality detection) needs another look before this router is trustworthy
enough to sit behind a single upload point.